In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics import accuracy_score,classification_report

In [22]:
dataset=pd.read_csv('../dataset/spam_ham.txt', sep='\t', names=['label', 'message'])

In [23]:
dataset

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [24]:
lemmatizer=WordNetLemmatizer()

In [25]:
# preprocessing

corpus=[]
for i in range(len(dataset)):
    rm_sep=re.sub('[^a-zA-Z]',' ',dataset['message'][i])
    words=word_tokenize(rm_sep)
    word=[lemmatizer.lemmatize(w) for w in words if w not in set(stopwords.words('english'))]
    corpus.append(' '.join(word))



In [26]:
corpus

['Go jurong point crazy Available bugis n great world la e buffet Cine got amore wat',
 'Ok lar Joking wif u oni',
 'Free entry wkly comp win FA Cup final tkts st May Text FA receive entry question std txt rate T C apply',
 'U dun say early hor U c already say',
 'Nah I think go usf life around though',
 'FreeMsg Hey darling week word back I like fun still Tb ok XxX std chgs send rcv',
 'Even brother like speak They treat like aid patent',
 'As per request Melle Melle Oru Minnaminunginte Nurungu Vettam set callertune Callers Press copy friend Callertune',
 'WINNER As valued network customer selected receivea prize reward To claim call Claim code KL Valid hour',
 'Had mobile month U R entitled Update latest colour mobile camera Free Call The Mobile Update Co FREE',
 'I gon na home soon want talk stuff anymore tonight k I cried enough today',
 'SIX chance win CASH From pound txt CSH send Cost p day day TsandCs apply Reply HL info',
 'URGENT You week FREE membership Prize Jackpot Txt word

In [27]:
y_label=pd.get_dummies(dataset['label']).astype('int')

In [28]:
y_label['spam']

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: spam, Length: 5572, dtype: int64

In [29]:
# train test split
X_train,X_test,y_train,y_test=train_test_split(
    corpus,y_label['spam'],test_size=0.2
)

In [30]:
X_test

['Is ok I stay night Xavier sleeping bag I getting tired',
 'Today ACCEPT DAY U Accept Brother Sister Lover Dear Best Clos Lvblefrnd Jstfrnd Cutefrnd Lifpartnr Belovd Swtheart Bstfrnd No rply mean enemy',
 'Send number give reply tomorrow morning said like ok',
 'And princess',
 'Aiyar sorry lor forgot tell u',
 'Wat time wan today',
 'And stop old man You get build snowman snow angel snowball fight',
 'What eat fo lunch senor',
 'Or go buy wif meet later',
 'Nokia phone lovly',
 'Sorry vikky Watching olave mandara movie kano trishul theatre wit frnds',
 'Just forced eat slice I really hungry tho This suck Mark getting worried He know I sick I turn pizza Lol',
 'Night ended another day morning come special way May smile like sunny ray leaf worry blue blue bay Gud mrng',
 'I cant pick phone right Pls send message',
 'Which channel',
 'Just wanted say holy shit guy kidding bud',
 'Watching tv lor Nice one like lor',
 'Frnd juz word merely relationship silent promise say I YOU Wherevr Whe

In [31]:
pd.DataFrame(y_test).value_counts()

spam
0       977
1       138
Name: count, dtype: int64

In [32]:
tfidf=TfidfVectorizer()

In [33]:
# vectorizatio using Bow
X_train_vector=tfidf.fit_transform(X_train)

In [34]:
X_train_vector.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4457, 6603))

In [35]:
X_test_vector=tfidf.transform(X_test)

In [36]:
X_test_vector.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1115, 6603))

In [37]:
mnb=MultinomialNB()

In [38]:
mnb.fit(X_train_vector,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [39]:
y_predicted=mnb.predict(X_test_vector)

In [40]:
accuracy=accuracy_score(y_pred=y_predicted,y_true=y_test)
cr=classification_report(y_pred=y_predicted,y_true=y_test)

print(f'accuracy: {accuracy}')
print(f'\n Classification Report: \n{cr}')

accuracy: 0.9757847533632287

 Classification Report: 
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       977
           1       0.99      0.81      0.89       138

    accuracy                           0.98      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.98      0.98      0.97      1115

